# Oil, energy, and equity-sector returns

Compare calendar weekly, monthly, quarterly, and annual changes in WTI crude oil and the Energy Select Sector SPDR (`XLE`) with same-period and next-period returns for the broad US market and all 11 S&P sectors.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from tyche_client import TycheClient

from dotenv import load_dotenv

load_dotenv(override=True)  # reads TYCHE_BASE_URL / TYCHE_API_KEY from a local .env file (see .env.example)
# override=True: without it, editing .env and re-running this cell in an already-running
# kernel silently keeps the OLD value, since load_dotenv() won't clobber an env var that
# a previous run of this same cell already set -- a real bug this caused once, not hypothetical.

client = TycheClient(
    os.getenv("TYCHE_BASE_URL", "http://localhost:8000"),
    os.getenv("TYCHE_API_KEY", "tyk_paste_your_key_here"),
)

## Configuration

WTI is the FRED daily West Texas Intermediate spot-price series. Returns use each period's last available observation; weeks end Friday. A future return means the immediately following week, month, quarter, or year; it is aligned with the current oil/energy change, so the final incomplete future observation is dropped. Results are descriptive, not causal or a trading rule.

In [ ]:
START = "2000-01-01"
DRIVERS = {"WTI": "WTI crude oil spot", "XLE": "Energy sector ETF"}
ASSETS = {
    "SPY": "US market",
    "XLB": "Materials", "XLC": "Communication services",
    "XLY": "Consumer discretionary", "XLP": "Consumer staples",
    "XLE": "Energy", "XLF": "Financials", "XLV": "Health care",
    "XLI": "Industrials", "XLK": "Technology",
    "XLRE": "Real estate", "XLU": "Utilities",
}
PERIODS = {
    "Weekly": pd.offsets.Week(weekday=4),
    "Monthly": pd.offsets.MonthEnd(),
    "Quarterly": pd.offsets.QuarterEnd(),
    "Annual": pd.offsets.YearEnd(),
}
PERIOD_TO_DATE = {"Weekly": "WTD", "Monthly": "MTD", "Quarterly": "QTD", "Annual": "YTD"}

## Retrieve and align prices

The existing client batches the ETF request. Forward-filling only bridges non-trading days and missing WTI observations; it does not fill backward before an ETF's inception.

In [ ]:
etf_prices = client.price_matrix(list(ASSETS), start=START)
wti = client.series("fred", "DCOILWTICO", start=START).rename("WTI")
prices = pd.concat([wti, etf_prices], axis=1).sort_index().ffill()

if prices.empty or prices["WTI"].dropna().empty or prices["SPY"].dropna().empty:
    raise ValueError("WTI and SPY history are required; check the API URL, key, and data coverage")

prices.tail()

## Build same-period and next-period comparisons

For each driver/asset pair, the table reports Pearson correlation and the asset's average return when the driver rose. Pairwise missing-value handling preserves the longest available history for each ETF. The `observations` count applies to the next-period comparison and is usually one smaller than the same-period sample.

In [ ]:
def compare_period(offset):
    returns = prices.resample(offset).last().pct_change(fill_method=None)
    rows = []
    for driver in DRIVERS:
        for ticker, asset in ASSETS.items():
            if driver == ticker:
                continue
            same = returns[[driver, ticker]].dropna()
            future = pd.concat(
                [returns[driver].rename("driver"), returns[ticker].shift(-1).rename("asset")],
                axis=1,
            ).dropna()
            rows.append({
                "driver": driver, "ticker": ticker, "asset": asset,
                "same_period_correlation": same[driver].corr(same[ticker]),
                "next_period_correlation": future["driver"].corr(future["asset"]),
                "avg_same_return_when_driver_up": same.loc[same[driver] > 0, ticker].mean(),
                "avg_next_return_when_driver_up": future.loc[future["driver"] > 0, "asset"].mean(),
                "observations": len(future),
            })
    return pd.DataFrame(rows)

comparisons = pd.concat(
    {name: compare_period(offset) for name, offset in PERIODS.items()},
    names=["frequency"],
).reset_index(level=0).reset_index(drop=True)

assert set(comparisons["frequency"]) == set(PERIODS)
assert comparisons.filter(like="correlation").stack().between(-1, 1).all()
comparisons.head()

## Tables

Read the two correlation columns side by side: the first measures co-movement during the oil/energy move, while the second asks whether that move was associated with the following period's return. The conditional averages express the same comparison in return units.

In [ ]:
for frequency in PERIODS:
    for driver, driver_name in DRIVERS.items():
        print(f"{frequency}: {driver_name}")
        table = (
            comparisons.query("frequency == @frequency and driver == @driver")
            .set_index(["ticker", "asset"])
            .drop(columns=["frequency", "driver"])
        )
        display(table.style.format({
            "same_period_correlation": "{:+.2f}",
            "next_period_correlation": "{:+.2f}",
            "avg_same_return_when_driver_up": "{:+.1%}",
            "avg_next_return_when_driver_up": "{:+.1%}",
        }).background_gradient(cmap="RdYlGn", subset=[
            "same_period_correlation", "next_period_correlation"
        ], vmin=-1, vmax=1))

## WTI return scatterplots

Each panel overlays the asset's return during the WTI move (blue circles) and in the following period (orange triangles). Solid lines are simple least-squares fits, panel titles report the corresponding correlations, and the dashed vertical line marks WTI's current WTD/MTD/QTD/YTD return. Axes are shared within each figure so slope and dispersion remain comparable across the market and sectors; extreme observations are retained.

In [ ]:
for frequency, offset in PERIODS.items():
    returns = prices.resample(offset).last().pct_change(fill_method=None)
    current_wti = returns["WTI"].dropna().iloc[-1]
    as_of = prices["WTI"].last_valid_index().date()
    fig, axes = plt.subplots(3, 4, figsize=(16, 11), sharex=True, sharey=True)
    for ax, (ticker, asset) in zip(axes.flat, ASSETS.items()):
        same = returns[["WTI", ticker]].dropna()
        future = pd.concat([
            returns["WTI"].rename("WTI"),
            returns[ticker].shift(-1).rename("asset"),
        ], axis=1).dropna()
        ax.scatter(same["WTI"], same[ticker], s=18, alpha=.30, color="#2563eb", label="same period")
        ax.scatter(future["WTI"], future["asset"], s=20, alpha=.30, marker="^", color="#f59e0b", label="next period")
        for frame, y, color in [(same, ticker, "#2563eb"), (future, "asset", "#f59e0b")]:
            if frame["WTI"].nunique() > 1:
                slope, intercept = np.polyfit(frame["WTI"], frame[y], 1)
                x = np.array([frame["WTI"].min(), frame["WTI"].max()])
                ax.plot(x, slope * x + intercept, color=color, linewidth=2)
        ax.axhline(0, color="0.75", linewidth=.7)
        ax.axvline(0, color="0.75", linewidth=.7)
        ax.axvline(
            current_wti, color="#111827", linestyle="--", linewidth=1.8,
            label=f"current {PERIOD_TO_DATE[frequency]} ({current_wti:+.1%})",
        )
        ax.set_title(
            f"{ticker} · {asset}\n"
            f"r now {same['WTI'].corr(same[ticker]):+.2f}  |  next {future['WTI'].corr(future['asset']):+.2f}",
            fontsize=10,
        )
        ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
        ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
        ax.grid(alpha=.15)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(.5, .965))
    fig.suptitle(f"{frequency} WTI return vs market and sector returns · through {as_of}", fontsize=16, y=1)
    fig.supxlabel(f"{frequency.lower()} WTI return")
    fig.supylabel(f"{frequency.lower()} market / sector return")
    plt.tight_layout(rect=(.02, .03, 1, .94))
    plt.show()

## Correlation overview

Bars make the sign change between contemporaneous and forward relationships easy to scan. Forward correlations near zero are evidence against treating the driver as a simple directional signal.

In [ ]:
for driver, driver_name in DRIVERS.items():
    fig, axes = plt.subplots(len(PERIODS), 2, figsize=(13, 15), sharex=True)
    for row, frequency in enumerate(PERIODS):
        data = comparisons.query("frequency == @frequency and driver == @driver").set_index("ticker")
        for col, (field, horizon) in enumerate([
            ("same_period_correlation", "same period"),
            ("next_period_correlation", "next period"),
        ]):
            values = data[field].sort_values()
            values.plot.barh(
                ax=axes[row, col],
                color=values.map(lambda value: "seagreen" if value >= 0 else "firebrick"),
            )
            axes[row, col].axvline(0, color="black", linewidth=0.8)
            axes[row, col].set_title(f"{frequency} — {horizon}")
            axes[row, col].set_ylabel("")
    fig.suptitle(f"{driver_name}: correlation with market and sector returns", y=1.01)
    plt.tight_layout()
    plt.show()